In [18]:
pip install pandas scikit-learn werkzeug sqlalchemy flask_admin flask_migrate flask_rq2 flask_compress dnachisel openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.


In [31]:
import time
from io import BytesIO
from datetime import datetime, UTC, timedelta
import traceback
import json
import io
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from werkzeug.exceptions import BadRequest
from pathlib import Path
import joblib

#from app.helpers.fold_storage_manager import FoldStorageManager
from app.helpers.sequence_util import (
    get_measured_and_unmeasured_mutant_seq_ids,
    get_loci_set,
    process_and_validate_evolve_input_files,
)
from app.helpers.jobs_util import (
    _live_update_tail,
    _psql_tail,
    try_run_job_with_logging,
)


def run_evolvepro(activity_file_path: str, embedding_file_paths: list, fold_sequence: str, invokation_id: int):
    """Run the evolvepro workflow using file paths."""
    if not Path(activity_file_path).exists():
        raise BadRequest(f"Activity file {activity_file_path} not found")
    for path in embedding_file_paths:
        if not Path(path).exists():
            raise BadRequest(f"Embedding file {path} not found")

    def run_evolvepro_with_logger(add_log):
        """Helper function to run evolvepro with a logger."""
        wt_aa_seq = fold_sequence

        # 1. Get the activity file.
        add_log(f"Getting the activity file {activity_file_path}")
        raw_activity_df = pd.read_excel(activity_file_path)

        # 2. Read and merge all embedding CSVs
        add_log(f"Reading {len(embedding_file_paths)} embedding files")
        embedding_dfs = []
        chunk_size = 10000  # Adjust based on memory constraints

        for path in embedding_file_paths:
            try:
                # Create chunks iterator
                chunks = pd.read_csv(path, chunksize=chunk_size)

                # Process each chunk
                path_dfs = []
                for chunk in chunks:
                    path_dfs.append(chunk)

                # Combine chunks for this path
                if path_dfs:
                    embedding_dfs.append(pd.concat(path_dfs, ignore_index=True))
            except Exception as e:
                add_log(f"Failed to read embedding file {path}: {e}")
                raise

        # Combine all embeddings
        raw_embedding_df = pd.concat(embedding_dfs, ignore_index=True)
        add_log(f"Found {raw_embedding_df.shape[0]} embeddings")

        # 3. Process the activity and embedding data.
        try:
            activity_df, embedding_df = process_and_validate_evolve_input_files(
                fold_sequence, raw_activity_df, raw_embedding_df
            )
        except Exception as e:
            add_log(f"Failed to process and validate input files: {e}")
            raise

        measured_mutants, unmeasured_mutants = (
            get_measured_and_unmeasured_mutant_seq_ids(activity_df, embedding_df)
        )
        add_log(
            f"{len(measured_mutants)} measured mutants and {len(unmeasured_mutants)} unmeasured mutants"
        )

        # 4. Fit the random forest model.
        add_log("Fitting the random forest model")
        try:
            X_train = np.vstack(
                [json.loads(x) for x in embedding_df.loc[activity_df.index].embedding]
            )
            y_train = activity_df.activity.to_numpy()
            model = RandomForestRegressor(
                n_estimators=100,
                criterion="friedman_mse",
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                min_weight_fraction_leaf=0.0,
                max_features=1.0,
                max_leaf_nodes=None,
                min_impurity_decrease=0.0,
                bootstrap=True,
                oob_score=False,
                n_jobs=None,
                random_state=1,
                verbose=0,
                warm_start=False,
                ccp_alpha=0.0,
                max_samples=None,
            )
            model.fit(X_train, y_train)
            add_log("Model fit complete")
        except Exception as e:
            add_log(f"Failed to fit the model: {e}")
            raise

        # 5. Predict activities for unmeasured mutants.
        add_log("Predicting activities for all mutants")
        try:
            all_mutants_embedding_array = np.vstack(
                [
                    json.loads(x)
                    for x in embedding_df.loc[
                        measured_mutants + unmeasured_mutants
                    ].embedding
                ]
            )
            print(all_mutants_embedding_array.shape)

            y_all_pred = model.predict(all_mutants_embedding_array)

            predicted_activity_df = pd.DataFrame(
                {
                    "seq_id": measured_mutants + unmeasured_mutants,
                    "predicted_activity": y_all_pred,
                }
            )
            predicted_activity_df.index = predicted_activity_df.seq_id
            predicted_activity_df["relevant_measured_mutants"] = (
                predicted_activity_df.seq_id.apply(
                    lambda seq_id: " ".join(
                        [
                            m
                            for m in measured_mutants
                            if get_loci_set(m) & get_loci_set(seq_id)
                        ]
                    )
                )
            )
            predicted_activity_df["actual_activity"] = predicted_activity_df.join(
                activity_df.groupby(level=0).activity.mean(), how="left"
            ).activity
            predicted_activity_df = predicted_activity_df.sort_values(
                "predicted_activity", ascending=False
            )
        except Exception as e:
            add_log(f"Failed to predict activities: {e}")
            raise

        # 6. Store model, visualizations, and predicted activities in storage manager.
        evolve_directory = Path("evolve") / Path(activity_file_path).stem
        add_log(
            f"Storing model, visualizations, and predicted activities in {evolve_directory}"
        )
        try:
            model_buffer = io.BytesIO()
            joblib.dump(model, model_buffer)
            serialized_model_binary_string = model_buffer.getvalue()

            predicted_activity_csv_str = predicted_activity_df.to_csv(index=False)

            fsm = FoldStorageManager()
            fsm.setup()

            fsm.storage_manager.write_file(
                invokation_id,
                str(evolve_directory / "model.joblib"),
                serialized_model_binary_string,
            )
            fsm.storage_manager.write_file(
                invokation_id,
                str(evolve_directory / "predicted_activity.csv"),
                predicted_activity_csv_str,
            )
        except Exception as e:
            add_log(f"Failed to store results: {e}")
            raise

    try_run_job_with_logging(run_evolvepro_with_logger, invokation_id)

In [139]:
import time
from io import BytesIO
from datetime import datetime, UTC, timedelta
import traceback
import json
import io
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from werkzeug.exceptions import BadRequest
from pathlib import Path
import joblib

#from app.helpers.fold_storage_manager import FoldStorageManager
from app.helpers.sequence_util import (
    get_measured_and_unmeasured_mutant_seq_ids,
    get_loci_set,
    process_and_validate_evolve_input_files,
)




def train_model(wt_aa_seq,raw_activity_df,raw_embedding_df,evolve_directory):
    activity_df, embedding_df = process_and_validate_evolve_input_files(
                wt_aa_seq, raw_activity_df, raw_embedding_df
            )
    measured_mutants, unmeasured_mutants = (
                get_measured_and_unmeasured_mutant_seq_ids(activity_df, embedding_df)
            )
    X_train = np.vstack(
                [json.loads(x) for x in embedding_df.loc[activity_df.index].embedding]
            )
    y_train = activity_df.activity.to_numpy()
    model = RandomForestRegressor(
        n_estimators=100,
        criterion="friedman_mse",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        min_weight_fraction_leaf=0.0,
        max_features=1.0,
        max_leaf_nodes=None,
        min_impurity_decrease=0.0,
        bootstrap=True,
        oob_score=False,
        n_jobs=None,
        random_state=1,
        verbose=0,
        warm_start=False,
        ccp_alpha=0.0,
        max_samples=None,
    )
    model.fit(X_train, y_train)
    try:
        all_mutants_embedding_array = np.vstack(
            [
                json.loads(x)
                for x in embedding_df.loc[
                    measured_mutants + unmeasured_mutants
                ].embedding
            ]
        )
        print(all_mutants_embedding_array.shape)
        y_all_pred = model.predict(all_mutants_embedding_array)
        predicted_activity_df = pd.DataFrame(
            {
                "seq_id": measured_mutants + unmeasured_mutants,
                "predicted_activity": y_all_pred,
            }
        )
        predicted_activity_df.index = predicted_activity_df.seq_id
        predicted_activity_df["relevant_measured_mutants"] = (
            predicted_activity_df.seq_id.apply(
                lambda seq_id: " ".join(
                    [
                        m
                        for m in measured_mutants
                        if get_loci_set(m) & get_loci_set(seq_id)
                    ]
                )
            )
        )
        predicted_activity_df["actual_activity"] = predicted_activity_df.join(
            activity_df.groupby(level=0).activity.mean(), how="left"
        ).activity
        predicted_activity_df = predicted_activity_df.sort_values(
            "predicted_activity", ascending=False
        )
    except Exception as e:
        print(f"Failed to predict activities: {e}")
        raise
    predicted_activity_df.reset_index(drop=True,inplace=True)
    predicted_activity_csv_path = evolve_directory / f"Round_{round_num}_predicted_activity.csv"
    print(f"Storing predicted activities in {predicted_activity_csv_path}")
    try:
        predicted_activity_df.to_csv(predicted_activity_csv_path, index=False)
    except Exception as e:
        print(f"Failed to store predicted activities: {e}")
        raise
    return predicted_activity_df
def evaluate_predictions(predicted_activity_df,exp_activity_df,round_activity_df,num_var,round_num,evolve_directory):
    try:
        predict = predicted_activity_df["actual_activity"].isna()
        extract_predict = predicted_activity_df[predict]
        #print(extract_predict)
        top_var = extract_predict.iloc[0:num_var]
        print(top_var)
        top_var_real = pd.merge(top_var,exp_activity_df,on='seq_id', how='inner')
        top_var_real = top_var_real[['seq_id','activity']]
        top_var_csv_path = evolve_directory / f"Round_{round_num}_top_variants.xlsx"
        top_var_real.to_excel(top_var_csv_path, index=False)
        next_round_activity = pd.concat([round_activity_df,top_var_real], ignore_index=True)
    except Exception as e:
        print(f"Failed to Evaluate Predictions")
        raise
    return next_round_activity
def read_exp_data(round_base_path, round_file_names_single, wt_fasta_path, round_file_names_multi=None):
    """
    Read and process experimental data from multiple files.

    Args:
    round_base_path (str): Base path to the data directory containing the excel files.
    round_file_names_single (list): List of single mutant round file names.
    wt_fasta_path (str): Path to the wild-type FASTA file.
    round_file_names_multi (list): List of multi mutant round file names.

    Returns:
    pd.DataFrame: Processed and concatenated experimental data
    """

    # Load experimental data
    all_experimental_data = []
    for round_file_name in round_file_names_single:
        experimental_data = load_experimental_data(round_base_path, round_file_name, wt_fasta_path, single_mutant=True)
        all_experimental_data.append(experimental_data)

    if round_file_names_multi is not None:
        for round_file_name in round_file_names_multi:
            experimental_data = load_experimental_data(round_base_path, round_file_name, wt_fasta_path, single_mutant=False)
            all_experimental_data.append(experimental_data)
    
    processed_dfs = []
    # Process each round's data
    for round_num, df in enumerate(all_experimental_data, start=1):
        df_copy = df.copy()
        
        # Set iteration for WT in first round, exclude WT from subsequent rounds
        if round_num == 1:
            df_copy.loc[df_copy['updated_variant'] == 'WT', 'iteration'] = 0
        else:
            df_copy = df_copy[df_copy['updated_variant'] != 'WT']
        
        df_copy.loc[df_copy['updated_variant'] != 'WT', 'iteration'] = round_num
        df_copy['iteration'] = df_copy['iteration'].astype(float)
        df_copy.rename(columns={'updated_variant': 'variant'}, inplace=True)
        
        processed_dfs.append(df_copy)

    # Combine all processed dataframes
    combined_df = pd.concat(processed_dfs, ignore_index=True)

    return combined_df
def evolve_simulation(wt_aa_seq,embeddings_path,exp_activity_file_path,num_var,round_num):
    exp_activity_df = pd.read_excel(exp_activity_file_path)
    raw_embedding_df = pd.read_csv(embeddings_path)
    raw_activity_df = exp_activity_df.sample(num_var)
    while not raw_activity_df['seq_id'].isin(raw_embedding_df['seq_id']).all():
        raw_activity_df = exp_activity_df.sample(num_var)
    print(raw_activity_df)
    evolve_directory = Path("evolve") / Path(exp_activity_file_path).stem
    evolve_directory.mkdir(parents=True, exist_ok=True)
    for i in range(1,(round_num+1)):
        exp_data=[]
        evolve_directory / f"Round_{round_num}_top_variants.xlsx"
        if i == 1:
            predicted_activity_df = train_model(wt_aa_seq,raw_activity_df,raw_embedding_df,evolve_directory)
            current_round_activity_df = evaluate_predictions(predicted_activity_df,exp_activity_df,raw_activity_df,num_var,i,evolve_directory)
            print(i)
            print(current_round_activity_df)
            
        else:
            predicted_activity_df = train_model(wt_aa_seq,current_round_activity_df,raw_embedding_df,evolve_directory)
            current_round_activity_df = evaluate_predictions(predicted_activity_df,exp_activity_df,current_round_activity_df,num_var,i,evolve_directory)
       
    
    

In [142]:
wt_aa_seq = 'MAKEDNIEMQGTVLETLPNTMFRVELENGHVVTAHISGKMRKNYIRILTGDKVTVELTPYDLSKGRIVFRSR'
activity_file_path = r'E:\ProgrammingProjects\foldy\backend\EvolveTest\kelsic_Round1.xlsx'
exp_activity_file_path = r'E:\ProgrammingProjects\foldy\backend\EvolveTest\Kelsic_Data_foldy.xlsx'
embeddings_path = r'E:\ProgrammingProjects\foldy\backend\EvolveTest\007426_embeddings_esmc_600m_wt_dms.csv'
num_var = 12
round_num = 5
evolve_simulation(wt_aa_seq,embeddings_path,exp_activity_file_path,num_var,round_num)

     seq_id  activity
1186   K64H   0.76750
9      Q10L   0.95200
737    N43V   0.78775
1092    D5P   0.99025
403    G29E   1.00100
679    M40Y   0.17400
396    N28T   0.78575
800    I47A   0.93450
1164   S63F   0.83100
1048   T58K   0.90450
1426    M9H   0.79500
395    N28S   0.84600
(1369, 1152)
Storing predicted activities in evolve\Kelsic_Data_foldy\Round_5_predicted_activity.csv
   seq_id  predicted_activity relevant_measured_mutants  actual_activity
3    T12N            0.877290                                        NaN
4     D5V            0.876075                       D5P              NaN
6    G29N            0.871895                      G29E              NaN
7    G29S            0.871668                      G29E              NaN
8    L14Q            0.869068                                        NaN
9     D5S            0.866388                       D5P              NaN
10   P18R            0.866115                                        NaN
11   E56A            0.866027